# Leer antes de ejecutar.


Para poder ejecutar este Notebook, es necesario instalar la  libería  Gensim, en esta se encuentra el modelo Word2Vec que se implementó, por lo anterior, hay que realizar lo siguiente:
1.   Ejecutar el primer bloque de código: "!pip install --upgrade --force-reinstall numpy gensim"
2.   Google Colab pedirá que se reinicie la sesión, acepta esta ventana.
3.  Una vez reiniciada la sesión, ejecutar el bloque de código  que importa las funciones necesarias "from gensim.models  import . . . " etc.

Para probar los resultados de los modelos entrenados, no es necesario ejecutar todo el código, solo las celdas siguientes:

1.   Cargar los conjuntos crudos, bloque debajo de la importación de funciones "from gensim.models  import . . . ".
2.   Ejecutar la celda llamada "Preprocesamiento y tokenización"
3. Ejecutar la celda llamada "Variables de entrenamiento y objetivo"
4.  Ejecutar la celda llamada " Vectorización BOW+Tf-idf"
5. Ejecutar la celda  llamada "Datos para Word2Vec" y "Funciones de pooling"
6. Ejecutar la celda de reentrenamiento sobre train y val
7.  Ejecutar las últimas dos celdas en la sección llamada "Tests


Las celdas y secciones que no fueron ejecutadas corresponden a las búsquedas en malla para cada modelo y su carga, guardado, etc.



# Librerias y data set

In [ ]:
!pip install --upgrade --force-reinstall numpy gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.2.2
    Uninstalling wrapt-2.2.2:
      Successfully uninstalled wrapt-2.2.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: smart_open
    Found existing installation: smart_open 8.0.0
    Uninstalling smart_open-8.0.0:
      Successfully uninstalled smart_open-8.0.0
  Attempting uninstall: scipy
    Found existing installation

In [ ]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

In [ ]:
import pandas as pd

# Conjuntos de datos de entrenamiento, validación y prueba
df = pd.read_csv("train.csv", delimiter=";")
df_val=pd.read_csv("validation.csv", delimiter=";")
df_test = pd.read_csv("test.csv", delimiter=";")

# Preprocesamiento y tokenización.

In [ ]:
from nltk.tokenize import TweetTokenizer
import regex as re

# Inicializar el tokenizador de NLTK
tokenizer = TweetTokenizer()

#Esta función recibe un Tweet
def preprocesado(text):
    # 1. Separar el texto en una lista de palabras/símbolos (tokens)
    tokens = tokenizer.tokenize(text)
    tokens_modificados = []

    # 2. Iterar sobre cada token
    for token in tokens:
        if re.match(r'@\w+', token):
            # Reemplazar menciones a usuarios
            tokens_modificados.append('__user_mention__')
        elif re.match(r'http\S+', token):
            # Reemplazar enlaces web
            tokens_modificados.append('__url__')
        elif re.match(r'#\S+', token):
            # Ignorar/eliminar los hashtags
            continue
        else:
            # Conservar el resto del texto convertido a minúsculas
            tokens_modificados.append(token.lower())

    return tokens_modificados

# 3. Aplicar la función de preprocesado a la columna 'text' de los 3 DataFrames
df["tokens_raw"] = df['text'].apply(preprocesado)
df_test["tokens_raw"] = df_test['text'].apply(preprocesado)
df_val["tokens_raw"] = df_val['text'].apply(preprocesado)

# Variables de entrenamiento  y objetivo

In [ ]:
# Entrenamiento
X_train = df["tokens_raw"]
X_train = [" ".join(i) for i in X_train]
# Prueba
X_test= df_test["tokens_raw"].tolist()
X_test = [" ".join(i) for i in X_test]
# Validación
X_val = df_val["tokens_raw"].tolist()
X_val = [" ".join(i) for i in X_val]

#Variables objetivo:
Y_train= df["account.type"]
Y_test= df_test["account.type"]
Y_val= df_val["account.type"]
train_labels = Y_train.tolist()
test_labels = Y_test.tolist()
val_labels=Y_val.tolist()

# Vectorizacion BOW+TD-IDF

In [ ]:
#Vectorización tf idf
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Se define  el vectorizador
vect = TfidfVectorizer(
    ngram_range=(1, 1),      # Solo palabras individuales
    max_features=25000,      # Vocabulario de 25,000 palabras
    dtype=np.float32
)

# Se  calcula la vectorización del vocabulario en el conjunto de entrenamiento
train_features = vect.fit_transform(X_train)

# Se aplica la vectorización a los otros dos conjuntos de datos
val_features = vect.transform(X_val)
test_features = vect.transform(X_test)

# Lista de palabras aprendidas
feature_list = vect.get_feature_names_out()

# Vectorizacion Word2Vec

## Funciones de pooling

In [ ]:
#Promedio simple
def average_sentence_vector(tokens, model, dim):
    vectors = []
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(dim)

#Concatenar max min y avg
def concatenate_pooling(tokens, model, dim):
    vectors = []
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        vectors = np.array(vectors)
        mean_vec = np.mean(vectors, axis=0)
        max_vec = np.max(vectors, axis=0)
        min_vec = np.min(vectors, axis=0)
        return np.concatenate([mean_vec, max_vec, min_vec])
    else:
        #La dimension aumenta por la concatenacion
        return np.zeros(dim * 3)

#Valores maximos
def max_sentence_vector(tokens, model, dim):
    vectors = []
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        return np.max(vectors, axis=0)
    else:
        return np.zeros(dim)

#Suma  de los vectores
def sum_sentence_vector(tokens, model, dim):
    vectors = []
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    if vectors:
        return np.sum(vectors, axis=0)
    else:
        return np.zeros(dim)

## Datos para entrenar Word2Vec

In [ ]:
# Wordvec necesita una lista  donde cada tweet es una lista con cadenas adentro
import pickle
w2v_model = Word2Vec.load("w2v_malla.model")
X_train_tokens = [tweet.split() for tweet in X_train]
X_test_tokens = [tweet.split() for tweet in X_test]
X_val_tokens=[tweet.split() for tweet in X_val]

# Malla para encontrar el mejor modelo Word2Vec con una regresión clásica

In [ ]:
import time
from itertools import product
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score,accuracy_score, classification_report

# Funciones de pooling
pooling = {
    "concatenate": concatenate_pooling,
    "average": average_sentence_vector,
    "max": max_sentence_vector,
    "sum": sum_sentence_vector}
#Malla de parámetros de Word2Vec
parametros_malla= {
    'vector_size': [150, 300],
    'window': [5, 7, 12],
    'sg': [0, 1],
    'min_count': [1, 2],
    'pooling': list(pooling.keys())}
#Configuraciones
configuraciones = list(product(
    parametros_malla['vector_size'],
    parametros_malla['window'],
    parametros_malla['sg'],
    parametros_malla['min_count'],
    parametros_malla['pooling']
))

resultados = []
mallatotal= len(configuraciones)
tiempo_inicial = time.time()

print(f"Hay  {mallatotal} configuraciones en la malla \n")

for i, (vector_size, window, sg, min_count, pooling_name) in enumerate(configuraciones, start=1):

    print(f" Iteracion actual {i}/{mallatotal}")
    print(f" Pooling: {pooling_name}")
    print(f" Parámetros Word2Vec: vector_size={vector_size}, window={window}, sg={sg}, min_count={min_count}")

    pooling_func = pooling[pooling_name]
    tiempo_actual=time.time()
    #Se entrena un modelo de Word2Vec para la configuración actual
    w2v_model = Word2Vec(
        sentences=X_train_tokens,
        vector_size=vector_size,
        window=window,
        sg=sg,
        min_count=min_count,
        seed=66,
        epochs=10
    )

    # Set de entrenamiento
    X_train_vec = np.array([pooling_func(tokens, w2v_model, vector_size)for tokens in X_train_tokens])
    # Set de validación
    X_val_vec = np.array([pooling_func(tokens, w2v_model, vector_size)for tokens in X_val_tokens])

    # Regresión donde se medirá el desempeño con los pesos aprendidos
    classifier = LogisticRegression(solver='lbfgs',penalty=None,max_iter=3000,random_state=66)
    classifier.fit(X_train_vec, train_labels)

    # Predicciones sobre el conjunto de validación
    val_preds = classifier.predict(X_val_vec)
    # Métrica accuracy
    val_accuracy = accuracy_score(val_labels, val_preds)
    # Métrica recall sobre la clase bot
    val_recall = recall_score(val_labels, val_preds, pos_label='bot')

    tiempo = time.time() - tiempo_actual
    tiempo_promedio = (time.time() - tiempo_inicial) / i
    tiempo_restante = tiempo_promedio * (mallatotal - i)
    print(f"Métrica Accuracy: {val_accuracy:.4f}")
    print(f" Tiempo de ejecución {tiempo:.2f} | Tiempo estimado restante: {tiempo_restante/60:.2f} min\n")

    # Se guardan los resultados de cada modelo
    resultados.append({
        'vector_size': vector_size,
        'window': window,
        'sg': sg,
        'min_count': min_count,
        'pooling': pooling_name,
        'val_accuracy': val_accuracy,
        'val_recall': val_recall
    })

#Selección del mejor modelo de acuerdo a Accuracy
mejor_accuracy = max(resultados, key=lambda x: x['val_accuracy'])
#Selección del mejor modelo de acuerdo a Recall
mejor_recall = max(resultados, key=lambda x: x['val_recall'])

print("Mejor Accuracy")
print(mejor_accuracy)
print("Mejor Recall")
print(mejor_recall)

In [ ]:
'''w2v_model = Word2Vec(Mejor Accuracy
{'vector_size': 300, 'window': 12, 'sg': 0, 'min_count': 2,
'pooling': 'concatenate', 'val_accuracy': 0.8123370981754996, 'val_recall': 0.8619791666666666}
Mejor Recall para bot
{'vector_size': 150, 'window': 7, 'sg': 0, 'min_count': 1,
 'pooling': 'sum', 'val_accuracy': 0.7910512597741095, 'val_recall': 0.8706597222222222})'''

# Word2Vec modelo final:

## Entrenamiento del modelo que se encontró en malla sobre conjunto de entrenamiento

In [ ]:
w2v_model = Word2Vec(
        sentences=X_train_tokens,
        vector_size=300,
        window=12,
        sg=0,
        min_count=2,
        seed=66,
        epochs=10
)
w2v_model.save("w2v_malla.model")

In [ ]:
import numpy as np
from scipy import sparse
X_train_vec = np.array([concatenate_pooling(tokens, w2v_model, 300) for tokens in X_train_tokens])
X_val_vec = np.array([concatenate_pooling(tokens, w2v_model, 300) for tokens in X_val_tokens])
X_test_vec = np.array([concatenate_pooling(tokens, w2v_model, 300) for tokens in X_test_tokens])
X_dev_vec= sparse.vstack([X_train_vec, X_val_vec])

# Ajuste  para  regresión con penalizacion

In [ ]:
import time
from itertools import product
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, classification_report

# Malla de parámetros para Regresión Logística
parametros_malla = {
    'penalizacion': ['l2', 'l1'],  # 'l1' solo soportado por 'liblinear'
    'C': [1/64, 1/32, 1/16, 1/8, 1/4, 1/2, 1, 2, 4, 8, 16, 32, 64]
}

# Configuraciones
configuraciones = list(product(
    parametros_malla['penalizacion'],
    parametros_malla['C']))

resultados = []
mallatotal = len(configuraciones)
tiempo_inicial = time.time()

print(f"Hay {mallatotal} configuraciones en la malla \n")

for i, (penalizacion, C) in enumerate(configuraciones, start=1):

    print(f" Iteración actual {i}/{mallatotal}")
    print(f" Parámetros Regresión Logística: penalizacion={penalizacion}, C={C}")
    tiempo_actual = time.time()

    try:
        # Se entrena el modelo con la configuración actual
        # (el argumento de scikit-learn se llama penalty)
        classifier = LogisticRegression(
            solver='liblinear',
            penalty=penalizacion,
            C=C,
            random_state=66,
            max_iter=3000
        )

        classifier.fit(X_train_vec, train_labels)

        # Predicciones sobre el conjunto de validación
        val_preds = classifier.predict(X_val_vec)

        # Métrica Accuracy
        val_accuracy = accuracy_score(val_labels, val_preds)
        # Métrica Recall
        val_recall = recall_score(val_labels, val_preds, pos_label='bot')

        tiempo = time.time() - tiempo_actual
        tiempo_promedio = (time.time() - tiempo_inicial) / i
        tiempo_restante = tiempo_promedio * (mallatotal - i)
        print(f"Métrica Accuracy: {val_accuracy:.4f}")
        print(f" Tiempo de ejecución {tiempo:.2f} | Tiempo estimado restante: {tiempo_restante/60:.2f} min\n")

        # Se guardan los resultados de cada modelo
        resultados.append({
            'penalizacion': penalizacion,
            'C': C,
            'val_accuracy': val_accuracy,
            'val_recall': val_recall
        })

    except Exception as e:
        # Se imprime el error para facilitar la depuración
        print(f"Error en penalizacion={penalizacion}, C={C}: {e}")

# Selección del mejor modelo de acuerdo a Accuracy
mejor_accuracy = max(resultados, key=lambda x: x['val_accuracy'])

# Selección del mejor modelo de acuerdo a Recall
mejor_recall = max(resultados, key=lambda x: x['val_recall'])

print("Mejor Accuracy")
print(mejor_accuracy)
print("Mejor Recall")
print(mejor_recall)

Hay 26 configuraciones en la malla 

 Iteración actual 1/26
 Parámetros Regresión Logística: penalizacion=l2, C=0.015625
Métrica Accuracy: 0.8093
 Tiempo de ejecución 9.35 | Tiempo estimado restante: 3.90 min

 Iteración actual 2/26
 Parámetros Regresión Logística: penalizacion=l2, C=0.03125
Métrica Accuracy: 0.8102
 Tiempo de ejecución 10.82 | Tiempo estimado restante: 4.03 min

 Iteración actual 3/26
 Parámetros Regresión Logística: penalizacion=l2, C=0.0625
Métrica Accuracy: 0.8110
 Tiempo de ejecución 14.31 | Tiempo estimado restante: 4.41 min

 Iteración actual 4/26
 Parámetros Regresión Logística: penalizacion=l2, C=0.125
Métrica Accuracy: 0.8106
 Tiempo de ejecución 18.64 | Tiempo estimado restante: 4.87 min

 Iteración actual 5/26
 Parámetros Regresión Logística: penalizacion=l2, C=0.25
Métrica Accuracy: 0.8123
 Tiempo de ejecución 19.94 | Tiempo estimado restante: 5.11 min

 Iteración actual 6/26
 Parámetros Regresión Logística: penalizacion=l2, C=0.5
Métrica Accuracy: 0.8102


In [ ]:
'''Mejor Accuracy: {'penalty': 'l2', 'C': 4, 'val_accuracy': 0.8132, 'val_recall': 0.8724}
Mejor Recall:   {'penalty': 'l2', 'C': 0.015625, 'val_accuracy': 0.8093, 'val_recall': 0.8837}'''

# Random forest

In [ ]:
import time
from itertools import product
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, classification_report

# Malla de parámetros para Random Forest
parametros_malla = {
    'n_estimators': [100, 300, 500],            # Número de árboles
    'max_depth': [5, 15, 30],                   # Profundidad del árbol
    'min_samples_split': [5, 10, 15, 20, 30],   # Mínimo de muestras para dividir
    'min_samples_leaf': [2, 5, 10]              # Mínimo de muestras por hoja
}

# Configuraciones
configuraciones = list(product(
    parametros_malla['n_estimators'],
    parametros_malla['max_depth'],
    parametros_malla['min_samples_split'],
    parametros_malla['min_samples_leaf']))

resultados = []
mallatotal = len(configuraciones)
tiempo_inicial = time.time()

print(f"Hay {mallatotal} configuraciones en la malla \n")

for i, (n_estimators, max_depth, min_split, min_leaf) in enumerate(configuraciones, start=1):

    print(f" Iteracion actual {i}/{mallatotal}")
    print(f" Parámetros RandomForest: "
          f"n_estimators={n_estimators}, "
          f"max_depth={max_depth}, "
          f"min_samples_split={min_split}, "
          f"min_samples_leaf={min_leaf}")

    tiempo_actual = time.time()

    try:
        # Se entrena el modelo con la configuración actual
        classifier = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_split,
            min_samples_leaf=min_leaf,
            random_state=66,
            n_jobs=2
        )

        classifier.fit(X_train_vec, train_labels)

        # Predicciones sobre el conjunto de validación
        val_preds = classifier.predict(X_val_vec)
        # Métrica Accuracy
        val_accuracy = accuracy_score(val_labels, val_preds)
        # Métrica Recall
        val_recall = recall_score(val_labels, val_preds, pos_label='bot')

        tiempo = time.time() - tiempo_actual
        tiempo_promedio = (time.time() - tiempo_inicial) / i
        tiempo_restante = tiempo_promedio * (mallatotal - i)
        print(f"Métrica Accuracy: {val_accuracy:.4f}")
        print(f" Tiempo de ejecución {tiempo:.2f} | "
              f"Tiempo estimado restante: {tiempo_restante/60:.2f} min\n")

        # Se guardan los resultados de cada modelo
        resultados.append({
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_split': min_split,
            'min_samples_leaf': min_leaf,
            'val_accuracy': val_accuracy,
            'val_recall': val_recall
        })

    except Exception as e:
        print(f"Error en combinación: "
              f"n_estimators={n_estimators}, "
              f"max_depth={max_depth}, "
              f"min_samples_split={min_split}, "
              f"min_samples_leaf={min_leaf}")

# Mejor Accuracy
mejor_accuracy = max(resultados, key=lambda x: x['val_accuracy'])
# Mejor recall
mejor_recall = max(resultados, key=lambda x: x['val_recall'])

print("Mejor Accuracy")
print(mejor_accuracy)
print("Mejor Recall")
print(mejor_recall)

Hay 135 configuraciones en la malla 

 Iteracion actual 1/135
 Parámetros RandomForest: n_estimators=100, max_depth=5, min_samples_split=5, min_samples_leaf=2
Métrica Accuracy: 0.8028
 Tiempo de ejecución 31.36 | Tiempo estimado restante: 70.04 min

 Iteracion actual 2/135
 Parámetros RandomForest: n_estimators=100, max_depth=5, min_samples_split=5, min_samples_leaf=5
Métrica Accuracy: 0.8028
 Tiempo de ejecución 28.37 | Tiempo estimado restante: 66.21 min

 Iteracion actual 3/135
 Parámetros RandomForest: n_estimators=100, max_depth=5, min_samples_split=5, min_samples_leaf=10
Métrica Accuracy: 0.8015
 Tiempo de ejecución 17.19 | Tiempo estimado restante: 56.41 min

 Iteracion actual 4/135
 Parámetros RandomForest: n_estimators=100, max_depth=5, min_samples_split=10, min_samples_leaf=2
Métrica Accuracy: 0.8023
 Tiempo de ejecución 17.17 | Tiempo estimado restante: 51.36 min

 Iteracion actual 5/135
 Parámetros RandomForest: n_estimators=100, max_depth=5, min_samples_split=10, min_sampl

In [ ]:
'''
Mejor Accuracy
{'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 15,
'min_samples_leaf': 2, 'val_accuracy': 0.8370981754995656, 'val_recall': 0.9366319444444444}
Mejor Recall
{'n_estimators': 500, 'max_depth': 15, 'min_samples_split': 5,
 'min_samples_leaf': 10, 'val_accuracy': 0.8344917463075586, 'val_recall': 0.9401041666666666}
'''

# SVC

In [ ]:
import time
from itertools import product
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, classification_report

# Parámetros
parametros_malla = {
    'kernel': ["linear", "poly", "rbf"],
    'C': [1, 10, 100]}
# Configuraciones
configuraciones = list(product(
    parametros_malla['kernel'],
    parametros_malla['C']))

resultados = []
mallatotal = len(configuraciones)
tiempo_inicial = time.time()

print(f"Hay {mallatotal} configuraciones en la malla \n")

for i, (kernel, C) in enumerate(configuraciones, start=1):

    print(f" Iteración actual {i}/{mallatotal}")
    print(f" Parámetros SVM: kernel={kernel}, C={C}")
    tiempo_actual = time.time()

    try:
        # Se entrena el modelo con la configuración actual
        classifier = SVC(
            kernel=kernel,
            C=C
        )
        classifier.fit(X_train_vec, train_labels)
        # Predicciones sobre validation
        val_preds = classifier.predict(X_val_vec)

        # Métrica Accuracy
        val_accuracy = accuracy_score(val_labels, val_preds)
        # Métrica Recall
        val_recall = recall_score(val_labels, val_preds, pos_label='bot')

        tiempo = time.time() - tiempo_actual
        tiempo_promedio = (time.time() - tiempo_inicial) / i
        tiempo_restante = tiempo_promedio * (mallatotal - i)
        print(f"Métrica Accuracy: {val_accuracy:.4f}")
        print(f" Tiempo de ejecución {tiempo:.2f} | "
              f"Tiempo estimado restante: {tiempo_restante/60:.2f} min\n")

        # Se guardan los resultados de cada modelo
        resultados.append({
            'kernel': kernel,
            'C': C,
            'val_accuracy': val_accuracy,
            'val_recall': val_recall})

    except Exception as e:
        # Se imprime el error para facilitar la depuración
        print(f"Error en kernel={kernel}, C={C}: {e}")

# Mejor Accuracy
mejor_accuracy = max(resultados, key=lambda x: x['val_accuracy'])
# Mejor Recall
mejor_recall = max(resultados, key=lambda x: x['val_recall'])

print("Mejor Accuracy")
print(mejor_accuracy)
print("Mejor Recall")
print(mejor_recall)

In [ ]:
#Resultados del modelo:
'''
Modelo: kernel=rbf, C=1 -> Accuracy: 0.8119   .8128
Modelo: kernel=rbf, C=10 -> Accuracy: 0.8267     .8306
Modelo: kernel=rbf, C=100 -> Accuracy: 0.8232    .8284

Modelo: kernel=linear, C=1 -> Accuracy: 0.8228    .8167
Modelo: kernel=linear, C=10 -> Accuracy: 0.8206   .8110
Modelo: kernel=linear, C=100 -> ###############

Modelo: kernel=poly, C=1 -> Accuracy: 0.8158      .8162
Modelo: kernel=poly, C=10 -> Accuracy: 0.8328     .8319
Modelo: kernel=poly, C=100-> Accuracy: 0.8254     .8284
'''

#Reentrenamiento en conjunto de train+validation

In [ ]:
import pickle
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from scipy import sparse

#Conjuntos de entrenamiendo y etiquetas
X_dev_bow= sparse.vstack([train_features, val_features]) #BOW+Tf-idf
X_dev_vec = np.concatenate([X_train_vec, X_val_vec]) #word2vec
y_dev = train_labels + val_labels # Etiquetas train+val



#Regresión L2 Word2vec
classifier = LogisticRegression(penalty='l2', solver='liblinear', C=4, random_state=66)
classifier.fit(X_dev_vec, y_dev)
filename = 'regresion_ridge_C=4_model.sav'
pickle.dump(classifier, open(filename, 'wb'))
#Random forest n_estimators= 100 word2vec
randomforest_100_model = RandomForestClassifier( n_estimators=100, max_depth=15, min_samples_split=15,min_samples_leaf=2,random_state=66,n_jobs=2)
randomforest_100_model.fit(X_dev_vec, y_dev)
filename = 'randomforest_100_model.sav'
pickle.dump(randomforest_100_model, open(filename, 'wb'))
#SVM poly C=10
svc_poly_c10_model = SVC(kernel='poly', C=10)
svc_poly_c10_model.fit(X_dev_vec, y_dev)
filename = 'svc_poly_C=10_model.sav'
pickle.dump(svc_poly_c10_model, open(filename, 'wb'))



#Entrenamiento de regresion sin penalizacion con vectorizacion BOW+Td-idf
regresion_clasica_BOW = LogisticRegression(solver="lbfgs", penalty=None, max_iter=3000)
regresion_clasica_BOW.fit(X_dev_bow, y_dev)
filename = 'regresion_clasica_BOW.sav'
pickle.dump(regresion_clasica_BOW, open(filename, 'wb'))

#Entrenamiento de regresion sin penalizacion con vectorizacion Word2Vec
regresion_clasica_w2v = LogisticRegression(solver="lbfgs", penalty=None, max_iter=3000)
regresion_clasica_w2v.fit(X_dev_vec, y_dev)
filename = 'regresion_clasica_w2v.sav'
pickle.dump(regresion_clasica_w2v, open(filename, 'wb'))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Tests

In [ ]:
#Esta función evalua un modelo dado  X y Y
def evaluar_modelo(modelo, test_X, test_Y):
    y_pred = modelo.predict(test_X)
    accuracy = accuracy_score(test_Y, y_pred)
    print(f"Accuracy: {accuracy:.4f}")
    print("Reporte:")
    print(classification_report(test_Y, y_pred, digits=4))

In [ ]:
import pickle
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

#modelos de clasificación entrenados
regresion_clasica_BOW = pickle.load(open('regresion_clasica_BOW.sav', 'rb'))
regresion_clasica_w2v = pickle.load(open('regresion_clasica_w2v.sav', 'rb'))
regresion_ridge_C4_model = pickle.load(open('regresion_ridge_C=4_model.sav', 'rb'))
randomforest_100_model = pickle.load(open('randomforest_100_model.sav', 'rb'))
svc_poly_C10_model = pickle.load(open('svc_poly_C=10_model.sav', 'rb'))


#Word2Vec y vectorización del modelo preentrenado.
w2v_model = Word2Vec.load("w2v_malla.model")
X_train_vec = np.array([concatenate_pooling(tokens, w2v_model, 300) for tokens in X_train_tokens])
X_val_vec = np.array([concatenate_pooling(tokens, w2v_model, 300) for tokens in X_val_tokens])
X_test_vec = np.array([concatenate_pooling(tokens, w2v_model, 300) for tokens in X_test_tokens])
X_dev_vec= sparse.vstack([X_train_vec, X_val_vec])

#Evaluación de modelos  en el conjunto de prueba

## Baselines de regresiones sin penalizar con vectorización BOW+Td-idf
evaluar_modelo(regresion_clasica_BOW, test_features, test_labels)
evaluar_modelo(regresion_clasica_w2v, X_test_vec, test_labels)

## Modelos con búsqueda en malla
evaluar_modelo(regresion_ridge_C4_model, X_test_vec, test_labels)
evaluar_modelo(randomforest_100_model, X_test_vec, test_labels)
evaluar_modelo(svc_poly_C10_model, X_test_vec, test_labels)

Accuracy: 0.7189
Reporte:
              precision    recall  f1-score   support

         bot     0.7166    0.7250    0.7208      1280
       human     0.7213    0.7128    0.7170      1278

    accuracy                         0.7189      2558
   macro avg     0.7190    0.7189    0.7189      2558
weighted avg     0.7189    0.7189    0.7189      2558

Accuracy: 0.8178
Reporte:
              precision    recall  f1-score   support

         bot     0.7850    0.8758    0.8279      1280
       human     0.8593    0.7598    0.8065      1278

    accuracy                         0.8178      2558
   macro avg     0.8222    0.8178    0.8172      2558
weighted avg     0.8221    0.8178    0.8172      2558

Accuracy: 0.8225
Reporte:
              precision    recall  f1-score   support

         bot     0.7892    0.8805    0.8323      1280
       human     0.8646    0.7645    0.8115      1278

    accuracy                         0.8225      2558
   macro avg     0.8269    0.8225    0.8219      2